In [1]:
"""
This scipr runs a stripped version of the DDDurban, where subsurface moisture states are drwn from a distribuiton.
In addition, extreme precipitation of specidfied duration and the temporal distribuion of that precipitation for shorter
timesteps than the duration are dran from a GPD distribution and a beta distribuion. Short timesseries a re simulated many times
so that a extremevalue distribiion of floods can be build
"""

using LsqFit
using Statistics
using Dates
using DataFrames
using Plots
using CSV
#using BlackBoxOptim
using Distributions

##Preprocessing routines
include("\\\\nve.no\\fil\\h\\HB\\HB-modellering\\DDDtestbenk\\DDD_Julia\\DDDFunctions\\Big2SmallLambda.jl")
include("\\\\nve.no\\fil\\h\\HB\\HB-modellering\\DDDtestbenk\\DDD_Julia\\DDDFunctions\\CeleritySubSurface.jl")   # Static celerity
include("\\\\nve.no\\fil\\h\\HB\\HB-modellering\\DDDtestbenk\\DDD_Julia\\DDDFunctions\\CeleritySubSurface1UH.jl") # Dynamic Celierty
include("\\\\nve.no\\fil\\h\\HB\\HB-modellering\\DDDtestbenk\\DDD_Julia\\DDDFunctions\\SingleUH.jl")
include("\\\\nve.no\\fil\\h\\HB\\HB-modellering\\DDDtestbenk\\DDD_Julia\\DDDFunctions\\SingleUH_SSinit.jl")
include("\\\\nve.no\\fil\\h\\HB\\HB-modellering\\DDDtestbenk\\DDD_Julia\\DDDFunctions\\SingleNormalUH.jl")
#include("\\\\nve.no\\fil\\h\\HB\\HB-modellering\\DDDtestbenk\\DDD_Julia\\DDDFunctions\\LayerEstimationDesign.jl")
include("\\\\nve.no\\fil\\h\\HB\\HB-modellering\\DDDtestbenk\\DDD_Julia\\DDDFunctions\\LayerEstimationDesignEmpSM.jl")
include("\\\\nve.no\\fil\\h\\HB\\HB-modellering\\DDDtestbenk\\DDD_Julia\\DDDFunctions\\GrvInputDistributionICap2022.jl")
include("\\\\nve.no\\fil\\h\\HB\\HB-modellering\\DDDtestbenk\\DDD_Julia\\DDDFunctions\\OFICap.jl")
include("\\\\nve.no\\fil\\h\\HB\\HB-modellering\\DDDtestbenk\\DDD_Julia\\DDDFunctions\\LayerCapacityUpdate.jl")
include("\\\\nve.no\\fil\\h\\HB\\HB-modellering\\DDDtestbenk\\DDD_Julia\\DDDFunctions\\LayerCapacityInit.jl")
include("\\\\nve.no\\fil\\h\\HB\\HB-modellering\\DDDtestbenk\\DDD_Julia\\DDDFunctions\\LayerUpdate.jl")
include("\\\\nve.no\\fil\\h\\HB\\HB-modellering\\DDDtestbenk\\DDD_Julia\\DDDFunctions\\LayerInitUrbanDesign.jl")
include("\\\\nve.no\\fil\\h\\HB\\HB-modellering\\DDDtestbenk\\DDD_Julia\\DDDFunctions\\RiverUpdate.jl")


#include("\\\\nve.no\\fil\\h\\HB\\HB-modellering\\DDDtestbenk\\DDD_Julia\\DDDFunctions\\DDDEventFloodDesignFixedP.jl")      # Static celerity
include("\\\\nve.no\\fil\\h\\HB\\HB-modellering\\DDDtestbenk\\DDD_Julia\\DDDFunctions\\DDDEventFloodDesignFixedPDynSM.jl")# Dynamic celerity
########################################################################################


t1= time_ns() #
#catchment = "152.4" 
#catchment = "26.20" 
#catchment = "19.107" 
#catchment = "234.13" 
#catchment = "2.279" 
#catchment = "12.70" 
#catchment = "28.11" 
#catchment = "62.5" 
#catchment = "56.10" #
#catchment = "148.2" #
catchment = "55.4" #
#catchment = "24.8" #
#catchment = "165.11" #
#catchment = "16.75"
#catchment = "33.22" #Vejle
#catchment = "POF1" 
#catchment = "POF11" 

# in !0 we have incresed overland flo celerity by 20 %

paramfile = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\Parameters\\",catchment,"\\Par_",
    catchment,"_3h(15h)_AAR_DynSM.csv")

#utfile  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Lilleelv\\simres_",
#   catchment,"_3h(12h)_AAR_FixedPDynSM_spes2.csv")
#utfile  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Skivika\\simres_",
#   catchment,"_2min(10min)_AAR_FixedPDynSM.csv")
utfile  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Royknes\\simres_",
    catchment,"_3h(15h)_AAR_FixedPDynSM.csv")
#utfile  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Sandsli\\simres_",
#    catchment,"_2min(14min)_AAR_FixedPDynSM.csv")
#utfile  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Kraakfoss\\simres_",
#    catchment,"_3h(18h)_AAR_FixedPDynSM.csv")
#utfile  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Mevatnet\\simres_",
#    catchment,"_3h(24h)_AAR_FixedPDynSM.csv")
#utfile  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Aardal\\simres_",
#    catchment,"_3h(15h)_AAR_FixedPDynSM.csv")
#utfile  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Fustvatn\\simres_",
#    catchment,"_3h(18h)_AAR_FixedPDynSM_PUB.csv")
#utfile  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Moska\\simres_",
#    catchment,"_3h(30h)_AAR_FixedPDynSM.csv")
#utfile  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Tannsvatn\\simres_",
#    catchment,"_3h(30h)_AAR_FixedPDynSM.csv")
#utfile  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Bulken\\simres_",
#    catchment,"_3h(30h)_AAR_FixedPDynSM.csv")
#utfile  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Lye2\\simres_",
#    catchment,"_10min(90min)_AAR_FixedPDynSM.csv")
#utfile  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Etna\\simres_",
#    catchment,"_3h(18h)_AAR_FixedPDynSM_high.csv")
#utfile  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Vaekkava\\simres_",
#    catchment,"_3h(30h)_AAR_FixedPDynSM.csv")
#utfile  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Vejle\\simres_",
#    catchment,"_3h(9h)_AAR_FixedPDynSM.csv")
#utfile  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Lillestrom\\simres_",
#    catchment,"_1h(4h)_AAR_FixedPDynSM.csv")
#utfile  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Lillestrom\\simres_",
#    catchment,"_2min(10min)_AAR_FixedPDynSM.csv")

#utfile2  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Lilleelv\\simres2_",
#   catchment,"_3h(12h)_AAR_FixedPDynSM_spes2.csv")
utfile2  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Royknes\\simres2_",
    catchment,"_3h(15h)_AAR_FixedPDynSM.csv")
#utfile2  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Skivika\\simres2_",
#    catchment,"_2min(10min)_AAR_FixedPDynSM.csv")
#utfile2  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Sandsli\\simres2_",
#    catchment,"_2min(14min)_AAR_FixedPDynSM.csv")
#utfile2  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Kraakfoss\\simres2_",
#    catchment,"_3h(18h)_AAR_FixedPDynSM.csv")
#utfile2  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Mevatnet\\simres2_",
#    catchment,"_3h(24h)_AAR_FixedPDynSM.csv")
#utfile2  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Aardal\\simres2_",
#    catchment,"_3h(15h)_AAR_FixedPDynSM.csv")
#utfile2  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Fustvatn\\simres2_",
#    catchment,"_3h(18h)_AAR_FixedPDynSM_PUB.csv")
#utfile2  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Moska\\simres2_",
#    catchment,"_3h(30h)_AAR_FixedPDynSM.csv")
#utfile2  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Tannsvatn\\simres2_",
#    catchment,"_3h(30h)_AAR_FixedPDynSM.csv")
#utfile2  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Etna\\simres2_",
#    catchment,"_3h(18h)_AAR_FixedPDynSM_high.csv")
#utfile2  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Lye2\\simres2_",
#    catchment,"_10min(90min)_AAR_FixedPDynSM.csv")
#utfile2  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Bulken\\simres2_",
#    catchment,"_3h(30h)_AAR_FixedPDynSM.csv")
#utfile2  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Vaekkava\\simres2_",
#    catchment,"_3h(30h)_AAR_FixedPDynSM.csv")
#utfile2  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Vejle\\simres2_",
#    catchment,"_3h(9h)_AAR_FixedPDynSM.csv")
#utfile2  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Lillestrom\\simres2_",
#    catchment,"_1h(4h)_AAR_FixedPDynSM.csv")
#utfile2  = string("\\\\nve.no\\fil\\h\\HM\\Interne Prosjekter\\UrbanDesignFlom\\utdata\\Lillestrom\\simres2_",
#    catchment,"_2min(10min)_AAR_FixedPDynSM.csv")


prm = CSV.read(paramfile,DataFrame,header=["Name", "val"], delim=';')    

# Parameters to be calibrated
#         OFP,         GscInt     persons       GPsh         ICapP
tprm = [prm.val[19], prm.val[22], prm.val[24], prm.val[28], prm.val[33]]

#Gshape, Gscale = Big2SmallLambda(prm.val[22], prm.val[22]) # GshInt and GscInt Converts integrated celerity to layers. 

NumSim = 2500

ext_precip = 300.0 # This max precipitation value of given duration.The Code will run hydrology NumSim times Røyknes has 200 mm,Sandsli has 50 mm
                  # and exceute for all precip values up to ext_precip by each 0.5 mm 

function calib_wrapper_model(tprm, prm, utfile, utfile2, NumSim, ext_precip)
 maxP = DDDEventFloodDesign(tprm, prm, utfile, utfile2, NumSim, ext_precip)  
 return maxP 
end
    
calib_wrapper_model(tprm, prm, utfile, utfile2, NumSim, ext_precip) # a single run     

t2 = time_ns()
println("Time elapsed[s]= ",(t2-t1)/1.0e9)

   

M fra Subrutine 71.59364906360283
M fra Subrutine 71.59364906360283
Precip= 1.0
Precip= 1.5
Precip= 2.0
Precip= 2.5
Precip= 3.0
Precip= 3.5
Precip= 4.0
Precip= 4.5
Precip= 5.0
Precip= 5.5
Precip= 6.0
Precip= 6.5
Precip= 7.0
Precip= 7.5
Precip= 8.0
Precip= 8.5
Precip= 9.0
Precip= 9.5
Precip= 10.0
Precip= 10.5
Precip= 11.0
Precip= 11.5
Precip= 12.0
Precip= 12.5
Precip= 13.0
Precip= 13.5
Precip= 14.0
Precip= 14.5
Precip= 15.0
Precip= 15.5
Precip= 16.0
Precip= 16.5
Precip= 17.0
Precip= 17.5
Precip= 18.0
Precip= 18.5
Precip= 19.0
Precip= 19.5
Precip= 20.0
Precip= 20.5
Precip= 21.0
Precip= 21.5
Precip= 22.0
Precip= 22.5
Precip= 23.0
Precip= 23.5
Precip= 24.0
Precip= 24.5
Precip= 25.0
Precip= 25.5
Precip= 26.0
Precip= 26.5
Precip= 27.0
Precip= 27.5
Precip= 28.0
Precip= 28.5
Precip= 29.0
Precip= 29.5
Precip= 30.0
Precip= 30.5
Precip= 31.0
Precip= 31.5
Precip= 32.0
Precip= 32.5
Precip= 33.0
Precip= 33.5
Precip= 34.0
Precip= 34.5
Precip= 35.0
Precip= 35.5
Precip= 36.0
Precip= 36.5
Precip= 37.0
P

In [3]:
using Distributions
satur = 65.0
SMSh = 3.71
SMSc = 8.38
ext_precip = 10.0
g = Gamma(SMSh ,SMSc)  # Gamma distributed soilmoisture. We need the quantile to estimate the celerity
for i in 1: 1000
grvstate = rand(Gamma(SMSh ,SMSc),1) # Draw groundwater state [mm], ordinary distributions (not extreme),Gamma(shape ,scale) 
 println("grvstate, ext_precip =", grvstate, " ", ext_precip)
end    
#prob = cdf(g, satur)

grvstate, ext_precip =[31.503187760425348] 10.0


In [12]:
#function LayerEstimationDesign(GshInt,GscInt,Timeresinsec,maxDl,midDL, MAD, area2, NoL, gtcel)
 GshInt = 1.9                  # scalar float
 GscInt = 0.0058               # scalar float 
 midDL = 95.9                  # scalar float
 maxDl = 442                   # scalar float
 MAD = 6.46                    # scalar float
 Timeresinsec = 10800          # scalar float
 NoL = 2                       # scalar integer
 area2 = 108470000             # scalar float
 gtcel = 0.99                  # scalar float
 
mLam = GshInt*GscInt
varLam = GshInt*(GscInt)^2                           #Yevjevich p.145
meanIntk = mLam*midDL/Timeresinsec                   #mean celerity estimated through Integrated Celerity
antBox = Int(trunc(maxDl/(meanIntk*Timeresinsec)))+1 #Temporal length UH_MAD
UH_MAD = zeros(Float64,antBox)
sRes = zeros(Float64,antBox) # saturation sum

#Unit hydrograph for MAD
UH_MAD = SingleUH(meanIntk,Timeresinsec, midDL, maxDl, 0)

StSt = (1000*MAD*Timeresinsec)/(area2)         # Steady state Input eq. output in mm
sRes[1] = 0
sRes[2:antBox] .= StSt.*UH_MAD[2:antBox]

for i in 3: antBox
  sRes[i:antBox] .= sRes[i:antBox] + StSt.*UH_MAD[i:antBox]
end

mRes = sum(sRes)
Fact = mLam/mRes
stdRes = (varLam/Fact^2)^0.5                   # see Haan p.51

GshRes = mRes^2/stdRes^2
GscRes = stdRes^2/mRes

MLev = [1/(NoL-1):1/(NoL-1):1.0;]              # (sequence)Quantiles  to calculate reservoir levels [0.1:0.1:0.9;]

MLev[NoL-1] = gtcel                            # quantile for start overland flow
Res_prob = zeros(Float64,(NoL-1))
Magkap = zeros(Float64,NoL)
g = Gamma(GshRes,GscRes) 
#calculates the reservoir levels associated with quantiles. Mean is GshRes*GscRes
Res_prob .= quantile.(g,MLev)

#Capasity of Layers
ssRes1 = zeros(Float64,NoL)
ssRes1[1] = 2000                               # capacity of overland flow level

for i in 2:(NoL-1)
  ssRes1[i] = Res_prob[NoL-i+1]-Res_prob[(NoL-i)]
end

ssRes1[NoL] = Res_prob[1]                     # capasity for the first slowest level         
                       
Magkap = ssRes1                                 # capasity for Layers
M = Res_prob[(NoL-1)]                           # Total groundwater reservoir



println("Magkap fra Subrutine ", Magkap)
println("M fra Subrutine ", M)
println("GshRes fra Subrutine ", GshRes)
println("GscRes fra Subrutine ", GscRes)



Magkap fra Subrutine [2000.0, 187.948224727671]
M fra Subrutine 187.948224727671
GshRes fra Subrutine 1.8999999999999995
GscRes fra Subrutine 29.13516218348886
